In [2]:
import json
from pathlib import Path
from typing import List
from tqdm import tqdm

import requests

In [3]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]

In [4]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_ID = "qwen3:8b"
MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")

print(f"Ollama URL: {OLLAMA_BASE_URL}")
print(f"Model: {MODEL_ID}")

Ollama URL: http://127.0.0.1:11434
Model: qwen3:8b


In [ ]:
import re

SA_OUTPUT_TYPE = {
    "attack_ratio": "percentage",
    "attack_pattern": "short_text",
    "dominant_id": "can_id",
    "id_concentration": "short_text",
    "fps": "number",
    "traffic_shift": "short_text",
    "max_gap": "number",
    "timing_cause": "short_text",
    "dominant_var": "number",
    "payload_behavior": "short_text",
    "dlc_mode": "integer",
    "dlc_behavior": "short_text",
    "missing_count": "integer",
    "critical_behavior": "short_text",
    "out_of_range_id_count": "integer",
    "plausibility_relation": "short_text",
    "duplicate_count": "integer",
    "timestamp_cause": "short_text",
    "attack_type": "short_text",
    "evidence_pattern": "short_text",
}

SA_TYPE_HINT = {
    "percentage": "Answer with a percentage only, e.g., 12.5%.",
    "can_id": "Answer with one CAN ID only, uppercase hex format, e.g., 0x1AF.",
    "integer": "Answer with one non-negative integer only.",
    "number": "Answer with one numeric value only.",
    "short_text": "Answer with a short phrase only (1-4 words), not a full sentence."
}


def build_sa_prompt_text(context: str, question: str, output_type: str = "short_text", strict_mode: bool = False) -> str:
    """Short-answer prompt with type-only constraints (no fixed labels)."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis assistant. "
        "Read a CAN time-window log and answer one short-answer question. "
        "Think using timestamp patterns, ID frequency, gaps, DLC, payload behavior, flags, and baseline cues internally. "
        "Provide your answer in exactly two lines:\n"
        "Line 1: The direct answer only (brief, following the type hint).\n"
        "Line 2: A very short explanation (max 10 words) of why."
    )

    type_hint = SA_TYPE_HINT.get(output_type, SA_TYPE_HINT["short_text"])
    strict_tail = (
        "\nType enforcement: strictly follow the answer type hint only."
        if strict_mode else ""
    )

    user_prompt = (
        "Analyze the following CAN bus time-window log and answer the question.\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer type hint:\n{type_hint}{strict_tail}\n\n"
        "Provide in exactly two lines:\n"
        "Line 1: Answer\n"
        "Line 2: Brief explanation (max 10 words)"
    )

    # Format as simple prompt for Ollama
    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    return full_prompt


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def _sanitize_answer(text: str) -> tuple[str, str]:
    """Returns (answer, explanation) tuple."""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    if not lines:
        return "", ""
    
    answer = lines[0].strip("\"'`")
    explanation = lines[1].strip("\"'`") if len(lines) > 1 else ""
    
    return answer, explanation


def _is_valid_answer_format(ans: str, output_type: str) -> bool:
    if not ans:
        return False

    if output_type == "can_id":
        return re.fullmatch(r"0x[0-9A-F]+", ans) is not None

    if output_type == "integer":
        return re.fullmatch(r"\d+", ans) is not None

    if output_type == "percentage":
        if not ans.endswith("%"):
            return False
        try:
            float(ans[:-1])
            return True
        except Exception:
            return False

    if output_type == "number":
        try:
            float(ans)
            return True
        except Exception:
            return False

    words = ans.split()
    return 1 <= len(words) <= 4


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def query_sa_llm(context: str, question: str, sa_type: str = "", max_new_tokens: int = 48) -> tuple[str, str]:
    """Returns (answer, explanation) tuple."""
    output_type = SA_OUTPUT_TYPE.get(sa_type, "short_text")

    prompt_text = build_sa_prompt_text(context, question, output_type=output_type, strict_mode=False)
    payload = {
        "model": MODEL_ID,
        "prompt": prompt_text,
        "stream": False,
        "options": {
            "temperature": 0.3,
            "num_predict": max_new_tokens,
        },
    }
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        timeout=180,
    )
    resp.raise_for_status()
    completion = _extract_ollama_text(resp.json())
    ans, expl = _sanitize_answer(completion)

    if _is_valid_answer_format(ans, output_type):
        return ans, expl

    # Retry once with stricter type enforcement.
    retry_prompt = build_sa_prompt_text(context, question, output_type=output_type, strict_mode=True)
    retry_payload = payload.copy()
    retry_payload["prompt"] = retry_prompt
    retry_resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=retry_payload,
        timeout=180,
    )
    retry_resp.raise_for_status()
    retry_completion = _extract_ollama_text(retry_resp.json())
    retry_ans, retry_expl = _sanitize_answer(retry_completion)

    return (retry_ans, retry_expl) if retry_ans else (ans, expl)


def load_questions(path: Path) -> List[dict]:
    """Load questions from .json (list) or .jsonl."""
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [ ]:
for ds_name in SELECTED_DATASETS:
    q_path = SA_QUESTION_FILES[ds_name]
    if not q_path.exists():
        print(f"[WARN] Short-answer questions for {ds_name} not found at {q_path}, skip.")
        continue

    questions = load_questions(q_path)
    print(f"[INFO] {ds_name}: loaded {len(questions)} short-answer questions.")

    out_dir = q_path.parent.parent / "llm_answers"
    out_dir.mkdir(parents=True, exist_ok=True)
    ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"

    ans_path.write_text("", encoding="utf-8")

    with ans_path.open("a", encoding="utf-8") as f:
        for rec in tqdm(questions, desc=f"{ds_name} SA answering"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("ground_truth", rec.get("answer", ""))

            pred, pred_expl = query_sa_llm(context, question, sa_type=rec.get("sa_type", ""))

            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": pred,
                "llm_explanation": pred_expl,
                "ground_truth": gt,
                "is_exact_match": normalize_text(pred) == normalize_text(gt) if gt else False,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: SA answers saved to {ans_path}")

[INFO] DoS: loaded 916 short-answer questions.


DoS SA answering: 100%|██████████| 916/916 [2:20:53<00:00,  9.23s/it]  


[INFO] DoS: SA answers saved to DoS_sa_qa\llm_answers\dos_sa_answers_qwen3_8b.jsonl
[INFO] Fuzzy: loaded 959 short-answer questions.


Fuzzy SA answering: 100%|██████████| 959/959 [2:05:03<00:00,  7.82s/it]  


[INFO] Fuzzy: SA answers saved to Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_qwen3_8b.jsonl
[INFO] Gear: loaded 1110 short-answer questions.


Gear SA answering: 100%|██████████| 1110/1110 [2:38:17<00:00,  8.56s/it] 


[INFO] Gear: SA answers saved to Gear_sa_qa\llm_answers\gear_sa_answers_qwen3_8b.jsonl
[INFO] RPM: loaded 1155 short-answer questions.


RPM SA answering: 100%|██████████| 1155/1155 [2:46:50<00:00,  8.67s/it] 

[INFO] RPM: SA answers saved to RPM_sa_qa\llm_answers\rpm_sa_answers_qwen3_8b.jsonl
